In [ ]:
import pandas as pd
import numpy as np
from torch import neg
from operator import pos

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB as nb
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, log_loss
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import ConfusionMatrixDisplay

import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

In [ ]:
original_train_dataset = pd.read_csv("conversion_data_train.csv")
original_train_dataset = original_train_dataset[original_train_dataset["age"]<= 100]

cvm_train_dataset = original_train_dataset.copy()
X = cvm_train_dataset.drop(columns=["converted"])
y = cvm_train_dataset["converted"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# II/ Entrainement des modèles de ML : Version 0 : Dumb models

#### - Modèle 1 : Logistic Regression
#### - Modèle 2 : HistGradientBoostingClassifier
#### - Modèle 3 : XGBoost

### Modèle Numéro 1 : Logistic Regression

In [ ]:
# preprocessing 

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"] # new_user est à exclure car déjà un int64 ? 

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
x_train_lr = preprocessor.fit_transform(X_train)
x_test_lr = preprocessor.transform(X_test)

In [ ]:
# params = {"C": 1.0, "solver": "lbfgs", "max_iter": 100}

lr_model = LogisticRegression() # **params)

lr_model.fit(x_train_lr, y_train)
y_pred = lr_model.predict(x_test_lr)
y_proba = lr_model.predict_proba(x_test_lr)[:, 1]

f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
recall = recall_score(y_test, y_pred)

print(f"F1 Score: {f1}")
print(f"Precision-Recall AUC: {pr_auc}")
print(f"ROC AUC: {roc_auc}")
print(f"Recall: {recall}")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non converti (0)", "Converti (1)"]
)
disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)

ax.set_title("Matrice de confusion - régression logistique")
ax.set_xlabel("Prédiction")
ax.set_ylabel("Valeur réelle")
plt.tight_layout()
plt.show()

### Modèle Numéro 2 : HistGradientBoostingClassifier

In [ ]:
# preprocessing 

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"] # new_user est à exclure car déjà un int64 ? 

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

In [ ]:
x_train_hgbc = preprocessor.fit_transform(X_train)
x_test_hgbc = preprocessor.transform(X_test)

In [ ]:
hgbc_model = HistGradientBoostingClassifier()

hgbc_model.fit(x_train_hgbc, y_train)
y_pred = hgbc_model.predict(x_test_hgbc)
y_proba = hgbc_model.predict_proba(x_test_hgbc)[:, 1]

f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
recall = recall_score(y_test, y_pred)

print(f"F1 Score: {f1}")
print(f"Precision-Recall AUC: {pr_auc}")
print(f"ROC AUC: {roc_auc}")
print(f"Recall: {recall}")
print(classification_report(y_test, y_pred))

In [ ]:
cm_hgbc= confusion_matrix(y_test, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_hgbc,
    display_labels=["Non converti (0)", "Converti (1)"]
)
disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)

ax.set_title("Matrice de confusion - HGBClassifier")
ax.set_xlabel("Prédiction")
ax.set_ylabel("Valeur réelle")
plt.tight_layout()
plt.show()

### Modèle Numéro 3 : XGboost

In [ ]:
# preprocessing 

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"] # new_user est à exclure car déjà un int64 ? 

categorical_transformer = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
x_train_xgb = preprocessor_xgb.fit_transform(X_train)
x_test_xgb = preprocessor_xgb.transform(X_test)

In [ ]:
xgb_model = XGBClassifier()

xgb_model.fit(x_train_xgb, y_train)
y_pred = xgb_model.predict(x_test_xgb)
y_proba = xgb_model.predict_proba(x_test_xgb)[:, 1]

f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)
recall = recall_score(y_test, y_pred)

print(f"F1 Score: {f1}")
print(f"Precision-Recall AUC: {pr_auc}")
print(f"ROC AUC: {roc_auc}")
print(f"Recall: {recall}")
print(classification_report(y_test, y_pred))

In [ ]:
cm_xgb= confusion_matrix(y_test, y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=["Non converti (0)", "Converti (1)"]
)
disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)

ax.set_title("Matrice de confusion - XGBoost")
ax.set_xlabel("Prédiction")
ax.set_ylabel("Valeur réelle")
plt.tight_layout()
plt.show()

### Comparaison des performances des modèles trois modèles : 

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 16))

models = [
    ("Régression logistique", lr_model, cm, x_test_lr),
    ("HistGradientBoosting", hgbc_model, cm_hgbc, x_test_hgbc),
    ("XGBoost", xgb_model, cm_xgb, x_test_xgb),
]

for row, (model_name, model, confusion_mat, x_test_model) in enumerate(models):
    y_pred_model = model.predict(x_test_model)
    y_proba_model = model.predict_proba(x_test_model)[:, 1]

    # Matrice de confusion à gauche
    disp = ConfusionMatrixDisplay(
        confusion_matrix=confusion_mat,
        display_labels=["Non converti (0)", "Converti (1)"]
    )

    disp.plot(
        ax=axes[row, 0],
        cmap="Blues",
        values_format="d",
        colorbar=False
    )

    axes[row, 0].set_title(f"Matrice de confusion - {model_name}")
    axes[row, 0].set_xlabel("Prédiction")
    axes[row, 0].set_ylabel("Valeur réelle")

    # Tableau des scores à droite
    metrics = pd.DataFrame({
        "Score": [
            f1_score(y_test, y_pred_model),
            precision_score(y_test, y_pred_model),
            recall_score(y_test, y_pred_model),
            average_precision_score(y_test, y_proba_model),
            roc_auc_score(y_test, y_proba_model),
        ]
    }, index=[
        "F1",
        "Precision",
        "Recall",
        "PR-AUC",
        "ROC-AUC",
    ])

    axes[row, 1].axis("off")
    axes[row, 1].set_title(f"Scores - {model_name}", fontsize=12)

    table = axes[row, 1].table(
        cellText=metrics.round(3).values,
        rowLabels=metrics.index,
        colLabels=metrics.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 2)

plt.suptitle("Comparaison des modèles", fontsize=16)
plt.tight_layout()
plt.show()

#### =>> En version dumb model, le modèle le plus performant est le modèle HistGradientBoostingClassifier avec un F1 score de 0.765 et un recall de 0.692.:  
- Le modèle de Logistic regression affiche un F1 score de 0.764 et un recall de 0.685. 
- Le modèle HistGradientBoostingClassifier affiche un F1 score de 0.765 et un recall de 0.692. 
- Le modèle XGBoost affiche un F1 score de 0.764 et un recall de 0.691.

Étant donné La proximité des performances des modèles, il est difficile de choisir un modèle en particulier. Cependant, le modèle HistGradientBoostingClassifier semble légèrement meilleur que les autres modèles. Nous commencerons donc par ce modèle pour l'optimisation des hyperparamètres. Nous évaluerons ensuite les performances des autres modèles après optimisation de leurs hyperparamètres respectifs.